# Position Bias in LLM-Based Recommender Systems

Notebook runner for the experiment suite. Each section runs one experiment
with live API calls, shows its figure inline and prints the numbers reported
in the thesis.

The scripts in `experiments/` are the source of truth; this notebook only
calls their `main()` with the sample sizes used for the reported runs. Raw
data goes to `results/*.json`, figures to `plots/*.png`.

**Before running:** the folder `tfg_experiments/` must be next to this
notebook (or this notebook sits inside it), and `pip install -r
requirements.txt` must have been run.

## 0. Setup

In [ ]:
import sys, os, getpass
from pathlib import Path

# Works whether this notebook sits inside the project folder or next to it.
PROJECT = Path.cwd()
if not (PROJECT / "common.py").exists():
    PROJECT = PROJECT / "tfg_experiments"
PROJECT = PROJECT.resolve()
assert (PROJECT / "common.py").exists(), f"project not found at {PROJECT}"

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / "experiments"))
os.chdir(PROJECT)          # results/ and plots/ are created inside the project

# The six candidate papers, with the real citation counts Experiment 4 uses
# as its popularity cue.
os.environ["EXP_ITEMS_FILE"] = "papers.csv"

# Experiments 1-5, 7 and 8 run on Llama 3.3 70B via Together AI.
os.environ["EXP_PROVIDER"] = "together"
if not os.environ.get("TOGETHER_API_KEY"):
    # Prompted securely; never written into the notebook file.
    os.environ["TOGETHER_API_KEY"] = getpass.getpass("TOGETHER_API_KEY: ")

# Optional overrides, set BEFORE importing common:
# os.environ["EXP_TEMPERATURE"] = "0.7"
# os.environ["EXP_MIN_INTERVAL"] = "0.0"

import common
print("Provider:", common.PROVIDER, "| model:", common.MODEL,
      "| temperature:", common.TEMPERATURE,
      "| pacing:", common.MIN_INTERVAL, "s/call")

In [ ]:
from IPython.display import Image, display
import importlib, json

def show(png_name):
    display(Image(filename=str(PROJECT / "plots" / png_name)))

def load_results(name):
    return json.loads((PROJECT / "results" / f"{name}.json").read_text())

def run(module_name, *argv):
    """Run an experiment's main() with CLI-style arguments."""
    mod = importlib.import_module(module_name)
    importlib.reload(mod)
    sys.argv = [module_name, *map(str, argv)]
    mod.main()

## 1. Single choice

Six candidates shown, one asked for, reshuffled every trial. Which **slot**
gets chosen is the position analysis; which **paper** gets chosen is the
content analysis. Both are reported, because in this format they are
entangled.

In [ ]:
run("exp1_single_choice", "--trials", 30)

In [ ]:
show("exp1_position_rates.png")
show("exp1_item_rates.png")
r = load_results("exp1_single_choice")
print(f"position: chi2 = {r['chi2_position']:.2f}, p = {r['p_position']:.4f}")
print(f"content:  chi2 = {r['chi2_item']:.2f}, p = {r['p_item']:.2e}")

## 2. Listwise ranking

The model ranks all six in one response. Mean output rank per input slot
shows the shape of the effect; Kendall's tau across reshuffles shows how
much of the ordering depends on input order rather than content.

In [ ]:
run("exp2_listwise_ranking", "--trials", 60)

In [ ]:
show("exp2_mean_rank_by_slot.png")
r = load_results("exp2_listwise_ranking")
mr = r["mean_rank_by_slot"]
print(f"mean output rank, first slot vs last: {mr[0]:.2f} -> {mr[-1]:.2f}")
print(f"mean Kendall's tau across trial pairs: {r['kendall_tau_mean']:.3f}")

## 3. Pairwise swap

Every pair judged twice with only the order exchanged. Both presentations
carry identical information, so each flipped verdict is one order-driven
decision and no statistical model is needed to read it.

In [ ]:
run("exp3_pairwise_swap", "--rounds", 3)

In [ ]:
show("exp3_pairwise_swap.png")
r = load_results("exp3_pairwise_swap")
judged = r["consistent"] + r["inconsistent"]
print(f"flipped by swap: {r['inconsistent']}/{judged} "
      f"({(1 - r['consistency_rate']) * 100:.0f}%)")
print(f"reversals favouring the item shown first: "
      f"{r['primacy_reversals']}/{r['inconsistent']}")

## 4. Popularity cue

A real citation count attached to one rotating item. The aggregate uplift on
its own would say "no popularity bias"; the breakdowns the script prints
contradict it. The cue works mainly when the boosted item is shown **early**,
and only for papers already competitive on content.

In [ ]:
run("exp4_popularity_cue", "--trials", 50)

In [ ]:
show("exp4_popularity.png")
r = load_results("exp4_popularity_cue")
print(f"aggregate uplift from the cue: {r['uplift_vs_uniform'] * 100:+.1f} pp\n")
print("P(chosen | cue) by the slot the cue item occupied:")
for slot, v in sorted(r["by_cue_slot"].items(), key=lambda kv: int(kv[0])):
    print(f"  slot {'ABCDEF'[int(slot)]}: {v['won']}/{v['shown']} = "
          f"{v['rate'] * 100:5.1f}%")

## 5. Verbosity

One rotating item padded with text that adds words but no facts, so any
uplift is a pure form effect. The per-item table shows the effect is graded:
papers sitting at zero stay at zero even when padded.

In [ ]:
run("exp5_verbosity", "--trials", 40)

In [ ]:
show("exp5_verbosity.png")
r = load_results("exp5_verbosity")
print(f"P(chosen | long form) = {r['p_chosen_given_long']:.3f}  "
      f"(uplift {r['uplift_vs_uniform'] * 100:+.1f} pp)\n")
print("pick rate when padded vs when concise:")
for i, v in sorted(r["by_item"].items(), key=lambda kv: int(kv[0])):
    print(f"  paper #{int(i) + 1}: {v['long_rate'] * 100:5.1f}% padded  |  "
          f"{v['short_rate'] * 100:5.1f}% concise")

## 6. Does model capability help?

The same protocol as Experiment 1, run across three Claude capability tiers.
This section needs an Anthropic key; the other experiments do not.

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
run("exp6_model_comparison", "--trials", 30)

In [ ]:
show("exp6_model_comparison.png")
r = load_results("exp6_model_comparison")
print(f"{'Model':<20}{'Chi2':<9}{'p-value':<11}{'Entropy':<10}{'Uplift(pp)'}")
for m in r["models"]:
    print(f"{m['label']:<20}{m['chi2']:<9.2f}{m['p_value']:<11.4f}"
          f"{m['entropy']:<10.3f}{m['max_slot_uplift'] * 100:<+10.1f}")

## 7. Mitigation in single choice

Baseline, debias prompt, PSC and ATE on the same decisions. The question is
not only how much stability each buys, but what it costs to buy it.

In [ ]:
run("exp7_mitigation_single_choice", "--decisions", 35, "--k", 5)

In [ ]:
show("exp7_mitigation_single_choice.png")
r = load_results("exp7_mitigation_single_choice")
print(f"{'Strategy':<18}{'Entropy':<11}{'Modal agr.':<13}{'Calls'}")
for key, label in (("baseline", "Baseline"), ("debias_prompt", "Debias prompt"),
                   ("psc", f"PSC (K={r['K']})"), ("ate", "ATE (proposed)")):
    c = r[key]
    print(f"{label:<18}{c['entropy']:<11.3f}"
          f"{c['modal_agreement'] * 100:<13.0f}{c['calls_per_decision']:.2f}")
print(f"\nATE escalated on {r['ate']['escalated']}/{r['M']} decisions")

## 8. Mitigation in ranking

Two parts. The first runs the API-costing conditions (standard, PSC/Borda,
RISE) and saves every individual run. The second reconstructs ATE from that
saved data, across the whole threshold range, at no additional API cost, and
produces both figures.

This is the most expensive experiment in the suite: 2 x (1 + K + (K-1))
calls per trial.

In [ ]:
run("exp8_mitigation_ranking", "--trials", 30)

In [ ]:
run("exp8_ate_analysis")

In [ ]:
show("exp8_mitigation_comparison.png")
show("exp8_ate_threshold.png")
r = load_results("exp8_ate_analysis")
print(f"{'Threshold':<12}{'Escalated':<12}{'Calls':<9}{'PC':<9}{'vs PSC':<10}{'p'}")
for row in r["threshold_sweep"]:
    p = "identical" if row["p_vs_psc"] is None else f"{row['p_vs_psc']:.4f}"
    escalated = f"{row['escalated']}/{row['n']}"
    print(f"{row['threshold']:<12.1f}{escalated:<12}{row['avg_calls']:<9.1f}"
          f"{row['mean_pc']:<9.3f}{row['diff_vs_psc']:<+10.3f}{p}")

## 9. Comparing the three biases

Position, popularity and verbosity on a single scale: uplift over the
uniform baseline.

In [ ]:
run("compare_biases")
show("compare_bias_magnitudes.png")

---
Every number above comes from the API calls just made. Export `results/` as
the thesis appendix, and report the exact N, model and temperature, which
each JSON records under `_meta`.